# 02 - Pre Processing

After some brief EDA measures, we must then clean our dataset by extracting useful raw features, removing non-negotiable null/empty fields, feature categorization, removing duplicate data, and handling major outliers.

### 02-01 Raw Feature Extraction
__Let's start by only encorperting useful raw features as previously menioned in the EDA notebook.__

The only raw features we wish to keep are as follows:

- **summary**: ticket title or short task description
- **description**: longer issue details
- **priority.name**: priority category
- **labels**: assigned Jira labels
- **issuetype.description**: description of the issue type
- **issuetype.name**: issue type label
- **issuetype.subtask**: whether the issue is a subtask
- **resolutiondate**: timestamp used to derive duration
- **created**: timestamp used to derive duration
- **watches.watchCount**: number of watchers

In [3]:
import pandas as pd

ticket_df = pd.read_csv('../jira_ticket_dataset.csv')
pd.set_option('display.max_rows', None)

raw_feature_columns = [
    "summary",
    "description",
    "priority.name",
    "labels",
    "issuetype.description",
    "issuetype.name",
    "issuetype.subtask",
    "resolutiondate",
    "created",
    "watches.watchCount"
]

# Reassign dataframe to contain only specified raw features

ticket_df = ticket_df[raw_feature_columns]

ticket_df.head()

C:\Users\Omar\AppData\Local\Temp\ipykernel_15964\783723080.py:3: DtypeWarning: Columns (0: customfield_12310921, 1: issuetype.subtask) have mixed types. Specify dtype option on import or set low_memory=False.
  ticket_df = pd.read_csv('../jira_ticket_dataset.csv')


,summary,description,priority.name,labels,issuetype.description,issuetype.name,issuetype.subtask,resolutiondate,created,watches.watchCount
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,[],An improvement or enhancement to an existing f...,Improvement,False,2005-01-01 07:50:46,2005-01-01 07:47:50,0.0
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors:\n1- runConfigure and conf...,Blocker,[],A problem which impairs or prevents the functi...,Bug,False,2004-12-30 05:30:36,2004-12-25 22:50:30,1.0
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,[],A problem which impairs or prevents the functi...,Bug,False,2005-01-02 15:21:00,2005-01-01 13:52:46,0.0
3,LogHandler can only work in GlobalConfiguratio...,org.apache.axis.handlers.LogHandler in request...,Major,[],A problem which impairs or prevents the functi...,Bug,False,NaN,2005-01-02 19:13:37,0.0
4,Decoding of service is broken in org.apache.ax...,The following code assumes a lot of things:\n\...,Major,[],A problem which impairs or prevents the functi...,Bug,False,NaN,2005-01-03 03:34:52,1.0


#### 02-02 Handling missing / null values

There exists some non-negotiable raw features that **must** be present in order to continue further analysis such as:

- **summary**: The model cannot predict ticket duration without a summary.
- **created**: You cannot calculate task duration without a start date.
- **resolutiondate**: You cannot calculate task duration without an end date.

For the optional values, the dataset scan shows a few common patterns:

- **labels** is very often the literal string `[]`, which means no labels were assigned.
- **description** can be missing or blank, but the ticket can still be modeled from the summary and metadata.
- **priority.name**, **issuetype.description**, and **issuetype.name** are categorical/context fields, so missing values should become explicit `unknown_*` categories.
- **issuetype.subtask** is a boolean-like field, but it can be missing or mixed-type, so it should be normalized to a real boolean after filling.
- **watches.watchCount** is numeric engagement metadata, so missing values should default to `0`.

**Default value decisions:** use `no_description` for missing descriptions, `no_labels` for missing or empty labels, `unknown_priority` for missing priority, `unknown_issue_type_description` for missing issue type descriptions, `unknown_issue_type` for missing issue type names, `False` for missing subtask flags, and `0` for missing watch counts.

Therefore we must drop any null rows associated with these columns.

In [6]:
required_columns = ["created", "resolutiondate", "summary"]
empty_like_values = ["", "[]", "{}", "None", "none", "null", "nan"]

for column in ticket_df.select_dtypes(include="object").columns:
    ticket_df[column] = ticket_df[column].astype("string").str.strip()
    ticket_df[column] = ticket_df[column].replace(empty_like_values, pd.NA)

ticket_df[required_columns].isnull().sum()

C:\Users\Omar\AppData\Local\Temp\ipykernel_15964\852441008.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in ticket_df.select_dtypes(include="object").columns:


created           0
resolutiondate    0
summary           0
dtype: int64

In [ ]:
ticket_df = ticket_df.dropna(subset=required_columns)

# check null values of each column to check they are all non-null values

if ticket_df[required_columns].isnull().any().any():
    print("failed: null value detected for some non-negotiable feature")
else:
    print("success: all non-negotiable features possess non-null values")

In [ ]:
optional_default_values = {
    "description": "no_description",
    "priority.name": "unknown_priority",
    "labels": "no_labels",
    "issuetype.description": "unknown_issue_type_description",
    "issuetype.name": "unknown_issue_type",
    "issuetype.subtask": False,
    "watches.watchCount": 0,
}

ticket_df = ticket_df.fillna(value=optional_default_values)

# check for string boolean and numeric values

ticket_df["issuetype.subtask"] = (
    ticket_df["issuetype.subtask"]
    .astype("string")
    .str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
)

ticket_df["watches.watchCount"] = pd.to_numeric(
    ticket_df["watches.watchCount"], errors="coerce"
).fillna(0)

ticket_df.isnull().sum()

summary                  0
description              0
priority.name            0
labels                   0
issuetype.description    0
issuetype.name           0
issuetype.subtask        0
resolutiondate           0
created                  0
watches.watchCount       0
dtype: int64

#### 02-03 Common dataset themes

After normalizing missing and empty-like values, scan the kept raw features for common Jira themes. This helps confirm that default values are meaningful and gives direction for later feature engineering.

In [8]:
theme_summary = {
    "top_priorities": ticket_df["priority.name"].value_counts().head(10),
    "top_issue_types": ticket_df["issuetype.name"].value_counts().head(15),
    "subtask_distribution": ticket_df["issuetype.subtask"].value_counts(),
    "top_labels": ticket_df["labels"].value_counts().head(15),
    "watch_count_summary": ticket_df["watches.watchCount"].describe(),
}

theme_summary

{'top_priorities': priority.name
 Major               620206
 Minor               179391
 Critical             44155
 Blocker              34793
 Trivial              27202
 unknown_priority     12173
 Normal               11227
 P2                    6390
 Low                   6164
 P3                    2629
 Name: count, dtype: Int64,
 'top_issue_types': issuetype.name
 Bug                   484165
 Improvement           226247
 Sub-task               88110
 Task                   64059
 New Feature            51905
 Test                   10336
 Dependency upgrade      7176
 Wish                    5260
 Documentation           2011
 Question                2006
 Story                   1652
 Technical task          1037
 Technical Debt           727
 Epic                     670
 Umbrella                 577
 Name: count, dtype: Int64,
 'subtask_distribution': issuetype.subtask
 False    857567
 True      89147
 Name: count, dtype: int64,
 'top_labels': labels
 no_labels         

#### 02-04 Feature Categorization

Features are differentiated by multiple factors such as dtype, model learning interpretation, and grouping togther related features.

There will be 3 main feature categories as follows:

- **Text-Based Features**: Descriptive data such as a ticket's description or summary.
- **Numerical-Based Features**: Numerical data such as votes and watch count.
- **Date-Based Features**: Date time features that are used to compute the estimated duration range.


In [ ]:
text_cols = [
    "summary",
    "description",
    "labels"
]

categorical_cols = [
    "priority.name",
    "issuetype.name"
]

date_optional_cols = [
    "created_month",
    "created_weekday",
    "created_is_weekend"
]